# FLAG 2027 - Notebook 09: Attacking the gender shortcut (our research contribution)

Same-gender pairs are our weakest cells (dev 36-40). Two ideas, both tested on top of our best system (**bal75**: CLIP | ECAPA+WavLM, challenge + MAV-Celeb + VoxCeleb2, per-dataset normalization, 75% in-domain batches):

| Version | Idea |
|---|---|
| G1_leace_face | **LEACE** (Belrose et al., NeurIPS 2023): erase all *linear* gender information from the FACE features before matching |
| G2_leace_voice | the same for the VOICE features |
| G3_leace_both | both |
| G4_samegender_neg | in the loss, wrong pairs of the SAME gender count 3x (hard negatives) |

LEACE is fitted on TRAINING rows only (challenge + VoxCeleb2 have gender labels) and then applied to everything, including dev - no test statistics are used.

**Honesty rule:** a version only counts as a fairness improvement if the **same-gender** error goes down - not just the gap (no 'levelling down').

Reference (bal75) is reused from Notebook 08. About 60-90 min. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - load challenge, MAV-Celeb and VoxCeleb2 features + GENDER per row
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
VOXF = [dict(np.load(os.path.join(FEATDIR, f))) for f in sorted(os.listdir(FEATDIR)) if f.startswith('vox_') and f.endswith('.npz')]
VOX = {k: np.concatenate([v[k] for v in VOXF]) for k in ['spk', 'gender', 'clip', 'ecapa', 'wavlm_sv']}

def gcode(g):   # 1 = female, 0 = male, -1 = unknown
    g = str(g).strip().lower()
    return 1 if g.startswith('f') else (0 if g.startswith('m') else -1)
G_CH = np.array([gcode(id_gender[c]) for c in y])
G_VOX = np.array([gcode(g) for g in VOX['gender']])

# MAV-Celeb has no gender labels -> predict them from the face (CLIP) with a classifier trained on the
# labelled TRAINING data (challenge + VoxCeleb2), then take the majority vote per person (training data only)
lab_X = np.concatenate([FEAT['clip']['train'], VOX['clip'][G_VOX >= 0]])
lab_z = np.concatenate([G_CH, G_VOX[G_VOX >= 0]])
gclf = LogisticRegression(max_iter=3000, C=0.1).fit(lab_X / np.linalg.norm(lab_X, axis=1, keepdims=True), lab_z)
p = gclf.predict(MAV['clip'] / np.linalg.norm(MAV['clip'], axis=1, keepdims=True))
per_person = pd.Series(p).groupby(MAV['spk']).mean()
G_MAV = np.array([int(per_person[s] >= 0.5) for s in MAV['spk']])

G_ALL = np.r_[G_CH, G_MAV, G_VOX]
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
note('Rows: challenge %d | MAV-Celeb %d (gender PREDICTED per person: %d of %d people female) | VoxCeleb2 %d'
     % (len(y), len(MAV['spk']), int((per_person >= 0.5).sum()), len(per_person), len(VOX['spk'])))

In [ ]:
# Cell 3 - combined table, per-dataset normalization, LEACE, model, losses, validation
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(ecapa, wavlm):
    return np.concatenate([l2(ecapa), l2(wavlm)], 1).astype(np.float32)

N_CH, N_MAV, N_VOX = len(y), len(MAV['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_VOX, int)]
F_RAW = np.concatenate([face_of(FEAT['clip']['train']), face_of(MAV['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(FEAT['ecapa']['train'], FEAT['wavlm_sv']['train']), voice_of(MAV['ecapa'], MAV['wavlm_sv']), voice_of(VOX['ecapa'], VOX['wavlm_sv'])])
EXT = N_CH + np.arange(N_MAV + N_VOX)

def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch_stats = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch_stats = st
    return F, V, ch_stats

class LEACE:
    # Least-squares concept erasure (Belrose et al. 2023), closed form, for a binary concept z.
    # After erasure no LINEAR classifier can predict z better than chance from the features.
    def fit(self, X, z, eps=1e-4):
        X = X.astype(np.float64); self.mu = X.mean(0); Xc = X - self.mu
        S = Xc.T @ Xc / len(X) + eps * np.eye(X.shape[1])
        s, U = np.linalg.eigh(S)
        W = (U * s ** -0.5) @ U.T; Winv = (U * s ** 0.5) @ U.T
        sxz = Xc.T @ (z - z.mean()) / len(X)
        u = W @ sxz; u = u / (np.linalg.norm(u) + 1e-12)
        self.M = (Winv @ np.outer(u, u) @ W).astype(np.float32); self.mu = self.mu.astype(np.float32)
        return self
    def transform(self, X):
        return (X - (X - self.mu) @ self.M.T).astype(np.float32)

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)

def opl_loss(feat, lab, gen=None, w_same=1.0):
    # orthogonal projection loss; optionally wrong pairs of the SAME gender get weight w_same (hard negatives)
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    if gen is not None and w_same != 1.0:
        sg = ((gen[:, None] == gen[None, :]) & (gen[:, None] >= 0)).float()
        neg = neg * (1 + (w_same - 1) * sg)
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()   # HIGHER = same person
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('ready: rows', len(Y_ALL), '| people', len(set(Y_ALL)))

In [ ]:
# Cell 4 - training (bal75 recipe) with optional PER-DATASET LEACE and same-gender-weighted negatives
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS, IN_FRAC = 16, 8, 2.0, 1e-4, 100, 15, 60, 0.75

def prepare(all_tr, leace):
    # per-dataset normalization, then ONE LEACE eraser PER DATASET, each fitted on that dataset's TRAINING rows
    # (v1 of this notebook fitted one pooled eraser -> gender survived on the challenge data; see probe check)
    F, V, ch_stats = domain_normalize(all_tr)
    erasers = {}
    for d in np.unique(D_ALL[all_tr]):
        fit_rows = all_tr[(D_ALL[all_tr] == d) & (G_ALL[all_tr] >= 0)]
        alld = np.where(D_ALL == d)[0]
        if leace in ('face', 'both'):
            erasers[('face', d)] = LEACE().fit(F[fit_rows], G_ALL[fit_rows].astype(float))
            F[alld] = erasers[('face', d)].transform(F[alld])
        if leace in ('voice', 'both'):
            erasers[('voice', d)] = LEACE().fit(V[fit_rows], G_ALL[fit_rows].astype(float))
            V[alld] = erasers[('voice', d)].transform(V[alld])
    return F, V, ch_stats, erasers

def train_variant(tr_idx, leace='none', w_same=1.0, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch_stats, erasers = prepare(all_tr, leace)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    gtr = torch.tensor(G_ALL[all_tr], device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j], gtr[j], w_same)
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch_stats, erasers

N_VOXP = len(set(VOX['spk']))
REF = 'E3_vox%d_dn_bal75' % N_VOXP
# G1-G3 of the first run (pooled eraser, invalid) stay in the CSV for the record; the corrected versions are G1d-G3d
VARIANTS = {'G1d_leace_face': ('face', 1.0), 'G2d_leace_voice': ('voice', 1.0),
            'G3d_leace_both': ('both', 1.0), 'G4_samegender_neg': ('none', 3.0)}
CSV = os.path.join(RESDIR, 'nb09_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, (lc, ws) in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _, _ = train_variant(tr_idx, lc, ws, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-18s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min'
                  % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))

In [ ]:
# Cell 5 - results table (reference bal75 from Notebook 08) + does LEACE really remove gender? (probe check)
R8 = pd.read_csv(os.path.join(RESDIR, 'nb08_cv.csv')); R8 = R8[R8.variant == REF]
R = pd.concat([R8, pd.read_csv(CSV)], ignore_index=True)
R['overall'] = (R['std'] + R['gender']) / 2
per_seed = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().reset_index()
tab = per_seed.groupby('variant').agg(std=('std', 'mean'), gender=('gender', 'mean'),
                                      overall=('overall', 'mean'), seed_sd=('overall', 'std'))
tab['gap'] = tab['gender'] - tab['std']
show = [REF, 'G1_leace_face', 'G2_leace_voice', 'G3_leace_both'] + list(VARIANTS)
tab = tab.loc[[v for v in dict.fromkeys(show) if v in tab.index]].round(2)
note('GENDER-SHORTCUT METHODS (validation on held-out CHALLENGE people, 5-fold x 3 seeds)')
note('  G1-G3 = first run, ONE pooled eraser (gender NOT removed on challenge data -> invalid test)')
note('  G1d-G3d = corrected, one eraser PER DATASET')
note(tab.to_string())

from sklearn.model_selection import GroupKFold, cross_val_score
all_tr = np.concatenate([np.arange(N_CH), EXT])
F0, V0, _, _ = prepare(all_tr, 'none'); F1, V1, _, _ = prepare(all_tr, 'both')
majority = max(G_CH.mean(), 1 - G_CH.mean())
def probe(X):
    return cross_val_score(LogisticRegression(max_iter=2000, C=0.1), X[:N_CH], G_CH, groups=y, cv=GroupKFold(5)).mean()
note('Gender probe on challenge rows (people-disjoint; chance = majority class %.2f): face %.2f -> %.2f | voice %.2f -> %.2f after per-dataset LEACE'
     % (majority, probe(F0), probe(F1), probe(V0), probe(V1)))

In [ ]:
# Cell 6 - final model for the best NEW variant (only if it beats the reference) -> submission ZIP
LOWER_IS_SAME = True
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
new_tab = tab.loc[[v for v in VARIANTS if v in tab.index]]
BEST = new_tab['overall'].idxmin()
if new_tab.loc[BEST, 'overall'] < tab.loc[REF, 'overall'] or new_tab.loc[BEST, 'gender'] < tab.loc[REF, 'gender'] - 0.5:
    lc, ws = VARIANTS[BEST]
    ep = int(np.median(R[R.variant == BEST]['best_ep']))
    finals = []
    for s in SEEDS:
        m, _, _, _, ch_stats, erasers = train_variant(np.arange(N_CH), lc, ws, None, seed=s, max_epochs=ep)
        finals.append(m)
    OUT = '/content/sub_09_' + BEST
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang, fname in LANG_FILE.items():
            split = track + '_' + lang
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
            fx = ((face_of(FEAT['clip'][split]) - ch_stats[0]) / ch_stats[1]).astype(np.float32)                  # challenge TRAIN stats
            vx = ((voice_of(FEAT['ecapa'][split], FEAT['wavlm_sv'][split]) - ch_stats[2]) / ch_stats[3]).astype(np.float32)
            if ('face', 0) in erasers: fx = erasers[('face', 0)].transform(fx)      # challenge eraser, fitted on TRAIN rows
            if ('voice', 0) in erasers: vx = erasers[('voice', 0)].transform(vx)
            xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
            s_ = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
            out = -s_ if LOWER_IS_SAME else s_
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, out):
                    f.write('%s %.6f\n' % (pid, v))
    SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
    ZIP_PATH = os.path.join(SUBDIR, '09_' + BEST + '.zip')
    with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: %s (%d epochs)' % (ZIP_PATH[len(BASE):], ep))
else:
    note('No new variant beats the reference (overall or same-gender) - no ZIP built.')

LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log = log[~log.run.isin(['09_' + v for v in VARIANTS])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '09_' + v, 'change': 'bal75 + per-dataset leace=%s, same-gender neg weight=%s' % VARIANTS[v],
                     'seed': '1,2,3', 'cv_std': tab.loc[v, 'std'], 'cv_gender': tab.loc[v, 'gender'],
                     'cv_overall': tab.loc[v, 'overall'], 'codabench_dev': ''} for v in VARIANTS if v in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)